<a href="https://colab.research.google.com/github/sensiwise-ai/agentic-ai-workshop2026-labs/blob/main/Lab1_RAG_Pipeline_Day1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lab 1 · RAG on a real company handbook

This lab introduces **Retrieval Augmented Generation (RAG)**, a technique that enhances large language models (LLMs) by providing them with external, up-to-date, and domain-specific information. We'll use a subset of GitLab's public handbook as our knowledge base, simulating a real-world scenario where an organization's internal documents are used to build an intelligent assistant. The goal is to build an assistant that can accurately answer questions based *only* on these documents, cite its sources, and gracefully admit when it doesn't have the information.

We will also investigate different **chunking strategies** and **metadata filters** to see how they impact the quality of the retrieved information and the final answer.

Here's an overview of the RAG pipeline we'll implement:

```
index once:   pages → clean → chunk → embed → ChromaDB (+ metadata)
per question: question → [filter on metadata] → top-k chunks → similarity gate → prompt → cited answer
```

### Setting up the LLM

This section focuses on preparing our language model (LLM) environment. We will install the necessary libraries and load a pre-trained LLM that we will later augment with our document retrieval system.

### Installing necessary libraries and importing modules

Here, we install all the Python libraries required for this lab. These include `transformers` for working with pre-trained models, `accelerate` for efficient model loading, `chromadb` for our vector database, `pypdf` for parsing PDF documents, `sentence-transformers` for creating embeddings, and `langchain-text-splitters` for advanced text chunking. We then import essential modules from these libraries to be used in subsequent steps.

In [ ]:
# Install necessary libraries silently (-q) and upgrade them (-U).
# `transformers` for pre-trained models, `accelerate` for efficient model loading,
# `chromadb` for vector database, `pypdf` for PDF parsing, `sentence-transformers` for embeddings,
# `langchain-text-splitters` for text chunking.
!pip install -q -U transformers accelerate chromadb pypdf sentence-transformers langchain-text-splitters

# Import core Python modules.
import torch  # PyTorch library for deep learning operations
import json   # For working with JSON data
import re     # For regular expressions

# Import specific classes from the transformers library.
from transformers import AutoTokenizer, AutoModelForImageTextToText # AutoTokenizer loads the correct tokenizer for a model, AutoModelForImageTextToText loads the model itself.

### Loading the Language Model (LLM)

In this step, we load a pre-trained Large Language Model (LLM) and its associated tokenizer. The tokenizer is crucial for converting human-readable text into a format the model can understand (numerical tokens), and vice-versa. We'll use a `Qwen` model, which is a powerful yet efficient model suitable for Colab's free GPU resources. `torch.float16` is used to optimize memory usage and speed.

In [ ]:
# Define the ID of the pre-trained language model to be used.
# 'Qwen/Qwen3.5-4B' is a 4 billion parameter open model, approximately 9 GB in FP16 precision.
# It's suitable for a free T4 GPU instance in Colab.
MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")

# Load the tokenizer associated with the specified model ID.
# The tokenizer is responsible for converting text into numerical tokens that the model can understand.
tok = AutoTokenizer.from_pretrained(MODEL_ID)

# Load the pre-trained language model itself.
# `dtype=torch.float16` uses half-precision floating points for reduced memory usage and faster computation.
# `device_map="auto"` automatically distributes the model across available devices (e.g., CPU and GPU).
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

### Defining the `chat` function for interacting with the LLM

This `chat` function provides a convenient way to interact with our loaded LLM. It takes a list of messages (or a single string) as input, formats them according to the model's specific chat template, tokenizes them, and then generates a response. The `max_new_tokens` and `temperature` parameters allow us to control the length and creativity of the model's output, respectively.

In [ ]:
def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    # If the input `messages` is a single string, convert it into the expected list format.
    # The model expects a list of dictionaries, where each dictionary has a 'role' (e.g., 'user') and 'content'.
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]

    # Apply the model's specific chat template to format the messages into a single prompt string.
    # `tokenize=False` means it returns a string, not token IDs.
    # `add_generation_prompt=True` adds a token to signal the model to start generating.
    # `enable_thinking=False` (specific to some models) disables internal 'thinking' steps.
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)  # wrap in the model's own chat markup

    # Convert the formatted prompt string into token IDs and move them to the model's device (e.g., GPU).
    # `return_tensors="pt"` ensures the output is a PyTorch tensor.
    inputs = tok(prompt, return_tensors="pt").to(model.device)

    # Generate a response from the model.
    # `**inputs` unpacks the tokenized input.
    # `max_new_tokens` limits the length of the generated response.
    # `temperature=0.0` makes the output deterministic (less creative, more factual).
    out = model.generate(**inputs, max_new_tokens=max_new_tokens)

    # Decode the generated token IDs back into a human-readable string.
    # `out[0][inputs["input_ids"].shape[1]:]` slices the output to remove the input prompt tokens, keeping only the generated part.
    # `skip_special_tokens=True` removes special tokens like `<s>`, `</s>`, etc.
    # `.strip()` removes leading/trailing whitespace.
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # drop the prompt, keep only what was generated

### Testing the `chat` function with a sample question

Before proceeding, it's essential to verify that our LLM has been loaded correctly and that the `chat` function is working as expected. This step sends a simple, general question to the model to ensure basic functionality and responsiveness.

In [ ]:
# Call the 'chat' function with a sample question about being an LLM.
# This tests if the model is loaded correctly and can generate a basic response.
print(chat("What is it like to be an LLM? in one sentence"))

### Testing the LLM's knowledge on a specific question without RAG

This section demonstrates the inherent limitations of a standalone LLM when asked about specific, private, or domain-specific information it was not explicitly trained on. We will ask a question about GitLab's internal policy to observe how the LLM responds without any external context, which often results in generic answers, confident but incorrect 'facts', or a refusal to answer.

## 0 · The problem: the model has never read your documents

Large Language Models are trained on vast amounts of public data, but they lack specific knowledge about private or recently updated organizational documents. This means they cannot reliably answer questions about your company's internal policies, handbooks, or specific domain information. Even if a public page was part of their training data, there's no way to verify if their answer comes from that memory or is merely a plausible guess. This limitation highlights the critical need for Retrieval Augmented Generation (RAG) to ground the LLM's responses in verifiable external sources.

In [ ]:
# Ask the LLM a question related to specific company policy without providing any external context.
# This demonstrates the LLM's potential to guess, give a generic answer, or generate an unverifiable 'fact'
# if it wasn't specifically trained on this proprietary information.
print(chat("At GitLab, what is the daily limit for meals while travelling for work?"))          # a guess, a shrug, or an unverifiable "fact"|

### Fetching and preparing documents from GitLab's handbook

This marks the beginning of the RAG pipeline where we acquire the documents that will form our knowledge base. We fetch specific pages from the GitLab public handbook to use as our source material. This step simulates pulling documents from a company's internal wiki, SharePoint, or other document management systems.

## 1 · Load the documents — with metadata

In a real RAG system, documents are sourced from various systems (e.g., wikis, file shares). For each document, beyond its text content, we also extract and store **metadata**. This metadata (such as `department`, `country`, `updated` date, and `url`) is crucial because it allows us to filter and refine our search later, ensuring that only the most relevant documents are considered for answering a question. This becomes particularly important in Section 6, where we demonstrate how metadata can steer the retrieval process.

In [ ]:
# Import necessary libraries for data fetching and processing.
import requests            # For making HTTP requests to fetch data from APIs
import re                  # For regular expressions, used for text parsing
import pandas as pd        # For data manipulation and analysis, especially for displaying metadata
import concurrent.futures as cf # For concurrent execution (threading) to speed up fetching multiple pages
from urllib.parse import quote # For URL encoding, used to safely include file paths in URLs

# Define the base API URL for GitLab's handbook repository.
API = "https://gitlab.com/api/v4/projects/gitlab-com%2Fcontent-sites%2Fhandbook/repository"
# Define a specific Git commit hash to ensure consistent handbook content for all users.
REF = "df66e66b937d38c1ed4e3dd452927ddf01be58b0"      # one fixed commit of the handbook, so everyone sees the same text

# List of specific handbook pages (Markdown files) to fetch.
PAGES = """
people-group/time-off-and-absence/time-off-types.md
people-group/time-off-and-absence/leave-types.md
people-group/anti-harassment.md
people-group/contracts-probation-periods.md
people-group/acceptable-use-policy.md
people-group/general-onboarding/_index.md
people-group/offboarding/faq.md
people-policies/inc-usa.md
people-policies/leave-of-absence/us.md
people-policies/ireland-ltd/right-to-disconnect-policy.md
people-policies/france-sas/remote-work-charter.md
total-rewards/benefits/_index.md
total-rewards/benefits/parental-leave-toolkit.md
total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk.md
total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd.md
total-rewards/benefits/general-and-entity-benefits/bv-benefits-netherlands.md
total-rewards/benefits/general-and-entity-benefits/inc-benefits-us/_index.md
total-rewards/benefits/general-and-entity-benefits/pty-benefits-australia.md
total-rewards/benefits/general-and-entity-benefits/canada-corp-benefits.md
total-rewards/compensation/_index.md
total-rewards/stock-options.md
finance/expenses.md
finance/spending-company-money.md
finance/procurement/individual-use-software.md
finance/accounts-payable/corp-credit-cards.md
security/policies_and_standards/password-standard.md
security/policies_and_standards/data-classification-standard.md
security/policies_and_standards/physical-security-standard-for-company-assets.md
security/policies_and_standards/records-retention-deletion.md
security/policies_and_standards/token-management-standard.md
legal/anti-corruption-policy.md
legal/anti-fraud-policy.md
legal/anti-retaliation-policy.md
legal/policies/gifts-contributions.md
legal/acceptable-use-policy.md
business-technology/_index.md
communication/confidentiality-levels.md
hiring/referral-process.md
""".split() # Split the multi-line string into a list of page paths.

# Define a dictionary to map parts of file paths to department names.
# This metadata helps categorize documents by their organizational function.
DEPARTMENT = {"people-group": "People", "people-policies": "People", "total-rewards": "Total Rewards", "finance": "Finance", "security": "Security", "legal": "Legal", "business-technology": "IT", "communication": "Communication", "hiring": "Hiring"}
# Define a dictionary to map keywords in file paths to country names.
# This metadata allows for filtering documents by geographical applicability.
COUNTRY = {"benefits-uk": "UK", "ireland": "IE", "netherlands": "NL", "benefits-us": "US", "inc-usa": "US", "/us.md": "US", "india": "IN", "france": "FR", "australia": "AU", "canada": "CA"}

def fetch(page):
    """One handbook page -> {title, department, country, updated, url, text}. The metadata is what filters will use later."""
    # Construct the full path to the raw Markdown file.
    path = "content/handbook/" + page

    # Fetch the raw Markdown text of the page from the GitLab API.
    # `quote(path, safe='')` URL-encodes the path.
    # `params={"ref": REF}` specifies the commit hash to ensure a fixed version.
    text = requests.get(f"{API}/files/{quote(path, safe='')}/raw", params={"ref": REF}).text      # raw markdown at that commit

    # Fetch the last commit date for the page to get its 'updated' timestamp.
    # This indicates freshness. The date is sliced to 'YYYY-MM-DD'.
    updated = requests.get(f"{API}/commits", params={"path": path, "ref_name": REF, "per_page": 1}).json()[0]["committed_date"][:10]   # last edit date = freshness

    # Split the Markdown content into YAML front matter and the main body.
    # Handbook pages typically start with YAML metadata followed by '---'.
    front, body = text.split("\n---\n", 1)                                      # YAML front matter | markdown body

    # Extract and return a dictionary containing the page's metadata and its text.
    return {"title": re.search(r"^title:\s*\"?(.*?)\"?\s*$", front, re.M).group(1),   # title lives in the front matter
            "department": DEPARTMENT[page.split("/")[0]], # Derive department from the first part of the page path.
            "country": next((c for key, c in COUNTRY.items() if key in page), "global"),   # Find country keyword in path, default to 'global'.
            "updated": int(updated.replace("-", "")),                          # Convert date to integer (e.g., 20260615) for numerical comparison.
            "url": "https://handbook.gitlab.com/handbook/" + page.replace("/_index.md", "/").replace(".md", "/"),   # Construct the public URL for the page.
            "text": body} # Store the main content of the page.

# Use a thread pool executor to fetch pages concurrently for faster processing.
# `cf.ThreadPoolExecutor(8)` creates a pool with 8 worker threads.
with cf.ThreadPoolExecutor(8) as pool:                                          # 8 pages at a time, ~15 seconds
    # Map the 'fetch' function to each page path in the PAGES list.
    # `list()` collects the results into a list of document dictionaries.
    docs = list(pool.map(fetch, PAGES))

# Print the total number of pages fetched and the total character count.
print(len(docs), "pages,", sum(len(d["text"]) for d in docs), "characters")

# Display a sample of 8 documents' metadata using a Pandas DataFrame.
# `.drop(columns="text")` removes the 'text' column for a cleaner view of metadata.
# `.sample(8, random_state=1)` selects 8 random rows for reproducibility.
pd.DataFrame(docs).drop(columns="text").sample(8, random_state=1)               # look at the metadata, not the text

### Defining a `clean` function to preprocess document text

Before we can effectively chunk and embed our documents, we need to clean them. Raw text often contains noise like HTML tags, template shortcodes, or extraneous characters that can interfere with the embedding process and reduce the quality of retrieval. This `clean` function aims to remove such irrelevant markup, ensuring that only meaningful text is processed.

## 2 · Clean

Raw document pages frequently contain various forms of markup (e.g., template shortcodes, HTML tags, unnecessary URLs) that do not contribute to the semantic meaning of the text. This noise can dilute the effectiveness of our embeddings and unnecessarily consume valuable context space within the LLM's prompt. Therefore, a cleaning step is essential to strip away this extraneous information, leaving only the core textual content.

In [ ]:
def clean(text):
    # Remove template shortcodes (e.g., {{< example >}}), HTML comments (<!-- -->), and HTML tags (<tag>).
    # `re.S` makes '.' match newlines as well.
    text = re.sub(r"\{\{[<%].*?[%>]\}\}|<!--.*?-->|<[^>\n]+>", "", text, flags=re.S)    # template shortcodes, HTML comments and tags
    # Replace Markdown links ([link text](url)) with just the link text.
    text = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", text)                                 # [link text](url) -> link text
    # Replace multiple spaces/tabs with a single space and strip leading/trailing whitespace.
    return re.sub(r"[ \t]+", " ", text).strip() # collapse runs of spaces

# Iterate through each document and apply the cleaning function to its 'text' content.
for d in docs:
    d["text"] = clean(d["text"])

# Print the title of the first cleaned document and the first 1000 characters of its text.
# This allows for a quick check to see the effect of the cleaning process.
print(docs[0]["title"], "\n", docs[0]["text"][:1000])

### Defining functions for different chunking strategies

Documents are often too long to be processed as a single unit by an embedding model or an LLM, primarily due to context window limitations and the need for focused embeddings. Therefore, we split documents into smaller pieces, or 'chunks'. This section defines different strategies for how we can perform this chunking, each with its own advantages and disadvantages.

## 3 · Chunking — four strategies

Splitting documents into manageable 'chunks' is a critical design decision in RAG. A single document can contain many topics, and embedding it as one large piece dilutes its semantic representation. Conversely, chunks that are too small might lose important context. Here, we explore four different chunking strategies, each with a distinct approach to balancing precision and context:

| Strategy | Idea | Risk |
|---|---|---|
| **tiny** — fixed 100 chars | very precise matching | the answer gets cut in half |
| **medium** — fixed 300 chars | a few sentences per chunk | may split a table or list |
| **huge** — fixed 3,000 chars | lots of context per hit | one vector for many topics: similarity gets diluted |
| **sections** — split on headings, cap at 800 | follows the author's structure, heading travels with the text | sections vary wildly in length |

The choice of strategy significantly impacts retrieval quality, and we will evaluate these differences.

In [ ]:
def chunk_fixed(text, size=300):
    """Split into pieces of about `size` characters, on word boundaries."""
    pieces, current = [], ""
    # Iterate through each word in the input text.
    for word in text.split():
        # Check if adding the current word would exceed the `size` limit.
        if len(current) + len(word) > size:
            # If it exceeds, add the current accumulated chunk to `pieces` and start a new chunk.
            pieces.append(current.strip()); current = ""     # close the chunk, start a new one
        # Add the current word and a space to the `current` chunk.
        current += word + " "
    # After the loop, add any remaining `current` chunk to `pieces`.
    return pieces + [current.strip()]              # don't lose the last, partial chunk

def chunk_sections(text, max_size=800):
    """Split on markdown headings; keep the heading with its text; cut very long sections with chunk_fixed."""
    pieces = []
    # Split the text by Markdown headings (e.g., # Heading, ## Subheading).
    # `(?=#{1,4} )` is a positive lookahead assertion, splitting *before* a heading but keeping the heading in the section.
    for section in re.split(r"\n(?=#{1,4} )", text):         # lookahead: split before a heading, keep the heading
        # Extract the heading if the section starts with one, otherwise set it to an empty string.
        heading = section.split("\n")[0] if section.startswith("#") else ""
        # If a section is too long, use `chunk_fixed` to further split it.
        # Ensure each piece starts with its original heading.
        pieces += [p if p.startswith(heading) else heading + "\n" + p for p in chunk_fixed(section, max_size)]   # heading travels with every piece
    # Filter out very short pieces (scraps) that are unlikely to be meaningful.
    return [p for p in pieces if len(p) > 60]                # drop scraps too short to mean anything

### Storing chunking strategies in a dictionary

To easily manage and compare the different chunking methods, we store them in a Python dictionary. Each key in `STRATEGIES` represents the name of a chunking strategy, and its corresponding value is a function (a lambda function in this case) that applies that specific chunking logic to a given text. This structure allows us to easily iterate through and apply each strategy.

In [ ]:
# Define a dictionary to store different chunking strategies.
# Each value is a lambda function (anonymous function) that takes text as input and applies a specific chunking logic.
# This allows for easy switching and comparison of chunking methods.
STRATEGIES = {"tiny": lambda t: chunk_fixed(t, 100),   # Fixed-size chunks of ~100 characters.
              "medium": lambda t: chunk_fixed(t, 300),  # Fixed-size chunks of ~300 characters.
              "huge": lambda t: chunk_fixed(t, 3000), # Fixed-size chunks of ~3000 characters.
              "sections": chunk_sections}             # Chunks based on Markdown sections, with a max size of 800.

### Demonstrating different chunking strategies on a sample page

This section visually demonstrates how each defined chunking strategy processes a single document. By applying 'tiny', 'medium', 'huge', and 'sections' chunking to the same page, we can observe the resulting number of chunks and the content of the first chunk. This helps us understand the practical differences between the strategies and their impact on how information is segmented.

In [ ]:
# Select a specific document (page) by its title for demonstration purposes.
# This page will be chunked in different ways to show the various strategies.
page = next(d for d in docs if d["title"] == "GitLab Password Standards")       # one page, four ways to cut it

# Iterate through each defined chunking strategy.
for name, chunker in STRATEGIES.items():
    # Apply the current chunking function to the selected page's text.
    pieces = chunker(page["text"])
    # Print the strategy name, the number of chunks produced, and the first 100 characters of the first chunk.
    # This provides a quick visual comparison of how each strategy splits the text.
    print(f"{name:<9} {len(pieces):>3} chunks | first chunk: {pieces[0][:100]!r}")

### Embedding chunks and indexing them in ChromaDB

Once documents are cleaned and chunked, the next step in the RAG pipeline is to convert these text chunks into numerical representations called 'embeddings' (or vectors). These embeddings capture the semantic meaning of the text. We then store these embeddings, along with their associated metadata, in a specialized database called a vector database (ChromaDB in this case). This allows for efficient similarity search later on.

## 4 · Embed and index — one collection per strategy

Each cleaned and chunked piece of text is transformed into a fixed-size vector using a small, efficient embedding model (specifically, `all-MiniLM-L6-v2`). This model runs quickly on a GPU. These vectors are then stored in **ChromaDB**, an in-memory vector database for this lab (meaning data is lost when the runtime resets). Crucially, we create a separate ChromaDB collection for each chunking strategy. This allows us to compare their performance directly.

When searching, similar meanings will result in nearby vectors, regardless of the exact words used. We configure ChromaDB to use `cosine` similarity, which measures the angle between vectors, indicating similarity in direction rather than magnitude. Every chunk also inherits the rich metadata from its original page, enabling powerful filtering during retrieval.

In [ ]:
# Import the chromadb library and its SentenceTransformerEmbeddingFunction.
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction

# Initialize the embedding function using a pre-trained Sentence Transformer model.
# 'all-MiniLM-L6-v2' is a small, efficient model that converts text into 384-dimensional vectors.
# `device="cuda"` ensures the embedding process runs on the GPU for speed.
embedder = SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2", device="cuda")   # text -> 384-dim vector, on the GPU

# Initialize the ChromaDB client. By default, this creates an in-memory database.
# This means the data will be lost when the Colab runtime restarts.
client = chromadb.Client()                                                                      # in-memory DB: gone when the runtime restarts

def index(name, chunker):
    """Creates a ChromaDB collection, embeds document chunks, and adds them to the collection."""
    # Create a new collection (database table-like structure) in ChromaDB.
    # `name` is the unique identifier for the collection (e.g., 'tiny', 'medium').
    # `embedding_function=embedder` specifies the model to use for converting text to vectors.
    # `metadata={"hnsw:space": "cosine"}` sets the distance metric for similarity search to cosine similarity.
    # Cosine similarity measures the angle between vectors, indicating similarity in direction rather than magnitude.
    kb = client.create_collection(name, embedding_function=embedder, metadata={"hnsw:space": "cosine"})   # cosine: compare direction, not length

    # Iterate through each document in the `docs` list, keeping track of its index `n`.
    for n, d in enumerate(docs):
        # Apply the specified `chunker` function to the document's text to split it into smaller pieces.
        pieces = chunker(d["text"])
        # Add the chunks to the ChromaDB collection.
        # `ids`: Generate unique IDs for each chunk (e.g., "0-0", "0-1" for document 0, chunk 0 and 1).
        # `documents`: The actual text content of each chunk.
        # `metadatas`: Each chunk inherits the metadata (title, department, country, updated, url) of its parent document.
        # This allows for filtering searches based on these attributes later.
        kb.add(ids=[f"{n}-{i}" for i in range(len(pieces))], documents=pieces,        # ids must be unique within the collection
               metadatas=[{k: d[k] for k in ("title", "department", "country", "updated", "url")}] * len(pieces))   # every chunk inherits its page's metadata
    return kb

# Create a dictionary of knowledge bases (KBS) where each key is a chunking strategy name
# and the value is the ChromaDB collection indexed using that strategy.
# This loops through `STRATEGIES` and calls the `index` function for each.
KBS = {name: index(name, chunker) for name, chunker in STRATEGIES.items()}     # four collections, same documents, different cuts

# Print the count of chunks in each created knowledge base (collection).
# This shows how many pieces of information each chunking strategy generated.
{name: kb.count() for name, kb in KBS.items()}

### Defining a `search` function to query the ChromaDB

This `search` function is the core of our retrieval mechanism. When a user asks a question, this function first embeds the question using the same model used for chunk embeddings. It then queries the specified ChromaDB knowledge base to find the `k` most semantically similar chunks. Additionally, it supports metadata filtering (`where` parameter) to narrow down the search space to more relevant documents before computing similarity, which can significantly improve retrieval accuracy.

## 5 · Search by meaning — and measure which chunking works

This section focuses on evaluating the effectiveness of our retrieval process. The `search()` function translates a user's question into an embedding and then retrieves the `k` most semantically similar chunks from our ChromaDB, along with a similarity score (where 1 indicates identical meaning). We will first observe how different chunking strategies retrieve information for a single question. Then, to quantitatively assess performance, we will measure how frequently the known answers to a set of fourteen evaluation questions appear within the top-3 retrieved chunks for each strategy. This helps us understand which chunking approach best captures the relevant information.

In [ ]:
def search(kb, question, k=3, where=None):
    """[(similarity, metadata, chunk_text), ...] for the k most similar chunks; `where` filters on metadata first."""
    # Query the ChromaDB knowledge base (`kb`) with the user's `question`.
    # `query_texts=[question]` provides the text to embed and search for.
    # `n_results=k` specifies the number of top-k most similar results to retrieve.
    # `where=where` applies a metadata filter if provided, narrowing down the search space before similarity comparison.
    r = kb.query(query_texts=[question], n_results=k, where=where)    # the question is embedded with the same model as the chunks

    # Process the raw results from ChromaDB into a more readable format.
    # It returns a list of tuples: (similarity score, metadata dictionary, chunk text).
    # `1 - dist` converts distance (0=identical) to similarity (1=identical).
    # `round(..., 2)` rounds the similarity score to two decimal places.
    # `zip()` is used to combine the documents, metadatas, and distances from the query result.
    return [(round(1 - dist, 2), meta, doc) for doc, meta, dist in zip(r["documents"][0], r["metadatas"][0], r["distances"][0])]

### Demonstrating search with a sample question and different chunking strategies

Here, we use a specific question to visually compare the retrieval results produced by each of our chunking strategies. For each strategy, we perform a search and print the similarity score, the title of the document the retrieved chunk came from, and the beginning of the chunk's text. This allows us to get a qualitative understanding of how each chunking method impacts what information is considered most relevant to the query.

In [ ]:
# Define a sample question to demonstrate the search function across different chunking strategies.
q = "What is the daily limit for meals while travelling for work?"

# Iterate through each knowledge base (representing a different chunking strategy).
for name, kb in KBS.items():
    # Perform a search for the question, retrieving only the top 1 result.
    # Unpack the result into score, metadata, and text.
    score, meta, text = search(kb, q, 1)[0]
    # Print the strategy name, the similarity score, the title of the document the chunk came from (truncated),
    # and the first 500 characters of the chunk text. This helps compare how different chunking strategies
    # retrieve information for the same question.
    print(f"{name:<9} {score}  {meta['title'][:]:<30} | {text[:500]!r}")

### Defining evaluation questions and expected answers

To quantitatively assess the performance of our retrieval system, we define a list of specific questions. Each question in this `EVAL` list is paired with a keyword or phrase that must be present in a truly relevant retrieved chunk. This allows us to automatically check if our RAG system successfully retrieves the correct information for a variety of known queries, providing a benchmark for our chunking strategies and overall retrieval effectiveness.

In [ ]:
# Define a list of evaluation questions, each paired with a specific text snippet that should be present in the correct answer.
# This `EVAL` list will be used to quantitatively assess the retrieval performance of different chunking strategies.
# Each tuple contains: (question, expected_answer_keyword).
EVAL = [  # (question, text that must appear in the retrieved context)
    ("How many days of time off should I take at minimum each year?", "25 days"),
    ("What percentage of my salary does the company pay into the UK pension?", "4%"),
    ("Which private medical insurance provider covers team members in Ireland?", "VHI"),
    ("What is the daily limit for meals while travelling for work?", "$120"),
    ("How many weeks of paid parental leave do team members get?", "16 weeks"),
    ("What is the minimum password length?", "12 characters"),
    ("How long can a personal access token be valid before it expires?", "30 days"),
    ("Can I install Chrome extensions on my work laptop?", "Chrome Extensions"),
    ("What is the maximum value of a gift to a US government official per event?", "$20"),
    ("How do I return my laptop when I leave?", "prepaid shipping label"),
    ("How long are production change records retained?", "3 years"),
    ("Are interns eligible for referral bonuses?", "not eligible"),
    ("How many vacation days are Netherlands team members entitled to?", "20 vacation days"),
    ("What is the reimbursement limit for internet while travelling?", "$50"),
]

### Evaluating retrieval performance for each chunking strategy

This `evaluate` function assesses the retrieval performance of each chunking strategy *independently of the LLM*. It focuses solely on whether the correct answer keyword is found within the top `k` retrieved chunks. This step is critical because if the relevant information isn't retrieved, the LLM will not be able to generate a correct answer. The function calculates metrics such as the total number of chunks, average chunk size, context provided per question, the hit rate (how many answers were found), and the mean similarity of the top-1 retrieved chunk.

In [ ]:
def evaluate(kb, k=3):
    # This function evaluates RETRIEVAL only; the language model is not involved yet.
    # If the exact answer text isn't found in the retrieved chunks, no prompt can save it.

    # Count how many evaluation questions have their expected answer text present in the top-k retrieved chunks.
    # `any(...)` checks if the answer keyword (case-insensitive) is in any of the top-k chunk texts.
    found = sum(any(ans.lower() in text.lower() for _, _, text in search(kb, q, k)) for q, ans in EVAL)

    # Calculate the average chunk size for the current knowledge base.
    avg = sum(len(c) for c in kb.get()["documents"]) // kb.count()       # average chunk size

    # Return a dictionary of evaluation metrics:
    return {"chunks": kb.count(),                          # Total number of chunks in the knowledge base.
            "avg chars": avg,                               # Average character count per chunk.
            "context per question": avg * k,                # Total characters of context provided to the model for each question.
            "answer found in top-3": f"{found}/{len(EVAL)}", # Ratio of questions where the answer was found.
            "mean top similarity": round(sum(search(kb, q, 1)[0][0] for q, _ in EVAL) / len(EVAL), 2)} # Average similarity score of the top-1 retrieved chunk across all questions.

# Create a Pandas DataFrame to display the evaluation results for each chunking strategy.
# The dictionary comprehension iterates through `KBS` (knowledge bases) and calls `evaluate` for each.
# `.T` transposes the DataFrame to make strategy names as rows and metrics as columns for better readability.
pd.DataFrame({name: evaluate(kb) for name, kb in KBS.items()}).T

### Selecting the 'medium' chunking strategy for further use

Based on the evaluation table from the previous step, we observe the trade-offs of different chunking strategies. For instance, 'tiny' chunks might achieve high similarity but often split answers, leading to lower recall. 'Huge' chunks provide a lot of context but can dilute the semantic meaning of individual vectors. The 'medium' and 'sections' strategies typically offer a better balance between precision and recall.

For the remainder of this lab, we will proceed with the `medium` chunking strategy, as it demonstrated a good balance of finding answers with a reasonable amount of context in our evaluation. Remember that the optimal chunking strategy is highly dependent on your specific documents and questions, so always measure and iterate!

Read the table across: **tiny** chunks score the *highest similarity* yet find the *fewest answers* — the matching phrase is there but the number next to it fell into the neighbouring chunk. **huge** chunks carry about 8,000 characters of context per question, slow the model down, and still miss answers because one vector now stands for many topics. **medium** and **sections** find the most with a fraction of the context. None of this is a law of nature: **the right chunking depends on your documents and your questions — measure it like this, do not copy it.**

We continue with `medium` as it offers a robust balance, effectively retrieving relevant information without excessive context or fragmenting critical data. This choice will be used for all subsequent RAG operations.

In [ ]:
# Select the 'medium' chunking strategy's knowledge base for subsequent operations.
# Based on the evaluation, 'medium' is chosen as a good balance for retrieval performance.
kb = KBS["medium"]

### Demonstrating metadata filtering with country-specific questions

## 6 · Metadata filters — change *which documents compete*

While similarity search ranks chunks based on their semantic relevance to a query, **metadata filters** allow us to precisely control *which chunks are even considered* for the search. Metadata (like `department`, `country`, or `updated` date) is exact and computationally cheap to apply. By using `where` clauses with our search query, we can drastically improve retrieval accuracy by limiting the search to documents that are contextually appropriate for the user's question, for example, specific to their country or department.

In [ ]:
# Define a question to demonstrate metadata filtering based on country.
q = "Which private medical insurance provider do I get?"

# Perform a search without any metadata filter.
# This shows results from all countries, potentially mixing relevant information.
print("no filter   :", [(s, m["title"][:28]) for s, m, _ in search(kb, q)])              # every country's page competes

# Perform a search with a filter for 'country' equal to 'UK'.
# This restricts the search to documents specifically tagged for the UK.
print("country=UK  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "UK"})])

# Perform a search with a filter for 'country' equal to 'IE' (Ireland).
# This restricts the search to documents specifically tagged for Ireland.
print("country=IE  :", [(s, m["title"][:28]) for s, m, _ in search(kb, q, where={"country": "IE"})])

### Demonstrating metadata filtering with date and department criteria

This section further illustrates the power of metadata filtering by applying more complex criteria. We will demonstrate how to filter documents based on their `updated` date (e.g., retrieving only recently updated documents) and their `department`. This shows how combining multiple metadata attributes can precisely narrow down the search to highly relevant information, simulating real-world scenarios where answers need to be current or specific to an organizational unit.

In [ ]:
# Define a question to demonstrate metadata filtering based on date and department.
q = "How quickly must I submit an expense after buying something?"

# Perform a search without any metadata filter.
# Shows general results related to expense submission.
print("all pages              :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q)])

# Perform a search with a filter for documents updated since June 1, 2026.
# `{"$gte": 20260601}` means 'greater than or equal to' the specified date (in YYYYMMDD integer format).
print("updated since June 2026:", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"updated": {"$gte": 20260601}})])

# Perform a search with a filter for documents from the 'Finance' department.
print("Finance only           :", [(s, m["title"][:26], m["updated"]) for s, m, _ in search(kb, q, where={"department": "Finance"})])

# Demonstrate a search with a filter for the 'Legal' department, using a different question.
# This shows how a specific filter can lead to irrelevant results if the question doesn't match the filter's scope.
print("department=Legal       :", [(s, m["title"][:26]) for s, m, _ in search(kb, "Can I install Chrome extensions on my laptop?", where={"department": "Legal"})])

### Defining the `answer` function for generating grounded responses with citations

This `answer` function encapsulates the full Retrieval Augmented Generation (RAG) pipeline. It takes a user's question, retrieves relevant chunks from our knowledge base (optionally applying metadata filters), and then uses these chunks as context to prompt the LLM. The function is designed to generate responses that are *grounded* in the retrieved documents, ensuring accuracy and providing clear citations. It also includes a similarity threshold to prevent the LLM from hallucinating answers when no sufficiently relevant information is found.

Three effects worth remembering from using metadata filters: the **country filter turns an ambiguous question into a precise one** (without it, the US and Australian pages outrank the UK's); a **"recent pages only" filter silently dropped the best page** ("Spending Company Money" was last edited in January 2025) — filters hide as well as focus, so derive them from what the user actually needs; and the **wrong department filter leaves only irrelevant chunks** with tiny similarity — which is exactly what the gate in the next section catches.

## 7 · Generate a grounded answer — the whole pipeline in one function

This `answer` function brings together all the components of our RAG system. It follows a structured process to ensure that the LLM's response is accurate, relevant, and verifiable:

1.  **Retrieve (with optional filter)**: It first uses the `search` function to fetch the top `k` most relevant chunks from the `kb` (knowledge base), applying any specified metadata filters.
2.  **Check the best similarity (refuse junk)**: If no chunks are found, or the similarity of the top chunk falls below a predefined `threshold`, the function explicitly states that it lacks the information. This prevents the model from generating speculative or incorrect answers.
3.  **Paste the chunks into the prompt as numbered passages**: The retrieved chunks are formatted and included directly in the LLM's prompt. Each chunk is given a numerical citation (e.g., `[1]`, `[2]`), and its document title is included.
4.  **Ask for citations**: The prompt explicitly instructs the LLM to answer *only* from the provided context and to cite its sources using the numbered passages.
5.  **Append the real sources**: After the LLM generates its response, the function appends a list of actual sources, linking each citation number to the document title and its URL. This provides transparency and allows users to verify the information.

In [ ]:
def answer(question, k=3, where=None, threshold=0.35):
    # Retrieve top-k relevant chunks from the knowledge base, optionally applying metadata filters.
    hits = search(kb, question, k, where)

    # If no hits are found, or the similarity of the top hit is below a predefined `threshold`,
    # it means there's no sufficiently relevant information. In this case, return a refusal message.
    if not hits or hits[0][0] < threshold:                                   # nothing relevant -> do not let the model improvise
        return f"I don't have that information in the handbook. (best similarity {hits[0][0] if hits else 0})"

    # Format the retrieved chunks into a context string for the LLM.
    # Each chunk is prefixed with a numbered citation (e.g., '[1]') and its document title.
    context = "\n\n".join(f"[{i}] ({m['title']}) {text}" for i, (_, m, text) in enumerate(hits, 1))

    # Construct the prompt for the chat model.
    # The 'system' role instructs the model to answer *only* from the provided context and cite its sources.
    # The 'user' role provides the context and the actual question.
    reply = chat([{"role": "system", "content": "Answer ONLY from the context passages. Cite them like [1]. If the context does not contain the answer, say so."},
                  {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}])

    # Generate a list of sources, linking each citation number to the document title and URL.
    sources = "\n".join(f"[{i}] {m['title']} — {m['url']}" for i, (_, m, _) in enumerate(hits, 1))

    # Combine the LLM's reply with the generated sources.
    return f"{reply}\n\nSources:\n{sources}"

# Test the `answer` function with a sample question that should be answerable from the handbook.
print(answer("What is the daily limit for meals while travelling for work?"))

### Testing the `answer` function with various questions, including out-of-scope

This section demonstrates the robustness of our `answer` function. We will test it with a range of questions: one that requires metadata filtering (country-specific), an out-of-scope question (to verify the refusal mechanism), and a question with an incorrect metadata filter (to show how filtering can lead to refusal if no relevant documents match). This showcases the function's ability to provide cited answers for in-scope queries and to gracefully handle situations where information is unavailable or filtered out.

In [ ]:
# Test the `answer` function with a question that includes a metadata filter for 'country' (Ireland).
# This should provide a specific answer for Irish medical insurance.
print(answer("Which private medical insurance provider do I get, and who pays the premium?", where={"country": "IE"}), "\n")

# Test with an out-of-scope question (about the World Cup).
# The model should respond with "I don't have that information" because the similarity score will be below the threshold.
print(answer("Who won the 2022 World Cup?"), "\n")                                                    # out of scope -> refused, not invented

# Test with a question where the metadata filter is likely to exclude relevant documents ('Legal' department for Chrome extensions).
# This demonstrates how incorrect filtering can lead to the model refusing to answer.
print(answer("Can I install Chrome extensions on my laptop?", where={"department": "Legal"}))          # wrong filter -> refused, not invented

### Comparing similarity scores for in-scope and out-of-scope questions to determine threshold

Understanding the distribution of similarity scores is crucial for setting an effective `threshold` in our `answer` function. This threshold acts as a 'similarity gate,' allowing us to filter out irrelevant information. In this section, we'll calculate and compare the similarity scores for questions that are within the scope of our handbook documents (`EVAL`) versus those that are intentionally outside its scope (`OUT_OF_SCOPE`). This comparison helps us identify a clear separation, or 'gap,' where we can confidently set a threshold to refuse answers when the retrieved context is not sufficiently similar to the query, thus preventing hallucination.

Compare with step 0: same model, now correct, sourced and refusing what it cannot back up. **Where does 0.35 threshold in our function (answer) come from?** From measuring, not guessing: in-scope questions land clearly above out-of-scope ones, and the threshold goes in the gap. It changes with the embedding model, the chunking and your documents — re-measure whenever any of them changes. (Watch *"the capital of France"*: it scores close to the gate because a French policy exists. Thresholds catch junk, not every wrong match.)

The choice of the `0.35` threshold is not arbitrary; it's determined empirically by analyzing the similarity scores of known in-scope versus out-of-scope questions. This threshold helps ensure that the RAG system only provides answers when it finds sufficiently relevant context, preventing the LLM from generating responses based on poor or unrelated retrievals. It's a dynamic value that should be re-evaluated if the embedding model, chunking strategy, or document set changes.

In [ ]:
# Define a list of questions that are intentionally out-of-scope for the handbook documents.
# These will be used to compare similarity scores with in-scope questions.
OUT_OF_SCOPE = ["Who won the 2022 World Cup?", "How do I fix a flat bicycle tyre?", "Best pizza recipe?", "What is the capital of France?"]

# Calculate and print the similarity scores for in-scope evaluation questions.
# It retrieves the top-1 similarity score for each question in `EVAL` and sorts them.
print("in scope  :", sorted(search(kb, q, 1)[0][0] for q, _ in EVAL))

# Calculate and print the similarity scores for out-of-scope questions.
# It retrieves the top-1 similarity score for each question in `OUT_OF_SCOPE` and sorts them.
print("out scope :", sorted(search(kb, q, 1)[0][0] for q in OUT_OF_SCOPE))

### Defining a function to upload custom documents

This section provides an exciting opportunity to apply the entire RAG pipeline to your own documents! We define an `upload` function that allows you to easily bring your proprietary files (PDFs, text, markdown, CSVs) into this Colab environment. These documents will then be processed, indexed, and made available for querying, just like the GitLab handbook pages.

## 8 · 🗂️ Your turn: your own documents

This is your chance to experiment with the RAG system using your own data. You can upload 1-5 files (e.g., `.pdf`, `.txt`, `.md`, `.csv`) directly from your local machine. These files will be securely stored within this Colab session (and deleted when the session ends). The same robust `clean` → `chunk` → `embed` → `search` → `answer` pipeline will be applied to your documents, allowing you to become the ultimate judge of the answer quality. This provides a practical way to test how the RAG system performs on data highly relevant to your specific needs.

In [ ]:
# Import the `files` utility from `google.colab` to enable file uploads in Colab.
from google.colab import files
# Import `PdfReader` from `pypdf` for extracting text from PDF documents.
from pypdf import PdfReader

def upload():
    """Open the file picker; return {filename: text} for .pdf / .txt / .md / .csv files."""
    texts = {}
    # Open the file upload dialog and iterate through the uploaded files.
    for name, blob in files.upload().items():
        # Check if the uploaded file is a PDF.
        if name.lower().endswith(".pdf"):
            # Write the binary content of the PDF blob to a local file.
            open(name, "wb").write(blob)
            # Use PdfReader to extract text from each page of the PDF and join them.
            texts[name] = "\n".join(page.extract_text() or "" for page in PdfReader(name).pages)
        else:
            # For other file types (txt, md, csv), decode the blob directly into text.
            # `errors="ignore"` handles potential decoding issues gracefully.
            texts[name] = blob.decode("utf-8", errors="ignore")
    # Print a summary of the uploaded files and their character counts.
    print({k: f"{len(v)} chars" for k, v in texts.items()})
    return texts

### Uploading, cleaning, chunking, and indexing custom documents

After you've used the `upload()` function to select your files, this code block automates the entire process of integrating them into our RAG system. It first calls `upload()` to get your documents, then cleans any existing custom document collection to ensure a fresh start. A new ChromaDB collection, `my_documents`, is created specifically for your uploaded files. Each document is then cleaned, chunked using the 'sections' strategy, and added to `my_documents` with basic metadata. Finally, it confirms how many chunks were created from your files.

In [ ]:
# Prompt the user to upload their own documents and store the results in the `mine` variable.
mine = upload()

# Check if a ChromaDB collection named "my_documents" already exists.
# If it does, delete it to ensure a clean state for new uploads.
client.delete_collection("my_documents") if "my_documents" in [c.name for c in client.list_collections()] else None

# Create a new ChromaDB collection specifically for the user's uploaded documents.
# It uses the same embedder and cosine similarity space as the handbook documents.
my_kb = client.create_collection("my_documents", embedding_function=embedder, metadata={"hnsw:space": "cosine"})

# Iterate through each uploaded document (filename and text content).
for name, text in mine.items():
    # Clean the document text using the `clean` function.
    # Chunk the cleaned text using the 'sections' chunking strategy.
    pieces = chunk_sections(clean(text))
    # Add the chunks to the 'my_documents' ChromaDB collection.
    # Generate unique IDs for each chunk based on filename and chunk index.
    # Assign basic metadata to each chunk, such as title (filename), a generic department 'mine',
    # 'global' country, a placeholder 'updated' date, and the filename as URL.
    my_kb.add(ids=[f"{name}-{i}" for i in range(len(pieces))], documents=pieces,
              metadatas=[{"title": name, "department": "mine", "country": "global", "updated": 20260907, "url": name}] * len(pieces))

# Print the total number of chunks created from the user's uploaded files.
print(my_kb.count(), "chunks from your files")

### Setting the knowledge base to the custom documents

Once your custom documents have been uploaded, cleaned, chunked, and indexed into `my_kb`, this step is crucial. By reassigning the global `kb` variable to point to `my_kb`, we effectively switch the active knowledge base. This means that all subsequent calls to the `answer()` function will now query *your* documents instead of the GitLab handbook. This enables you to interactively ask questions about your uploaded content.

In [ ]:
# Reassign the `kb` variable to point to `my_kb`.
# This action switches the active knowledge base for the `answer()` function from the GitLab handbook
# to the user's recently uploaded and indexed documents. Any subsequent calls to `answer()` will now
# query the custom document collection.
kb = my_kb                                                                       # point answer() at your documents

### Interactive loop to ask questions from custom documents

This code block provides an interactive command-line interface for querying your newly indexed custom documents. The `while True` loop continuously prompts you to `Ask your documents:` for questions. Your input is then passed to the `answer()` function, which retrieves relevant information from `my_kb` and generates a grounded, cited response. You can type `quit`, `exit`, `q`, or simply press Enter on an empty prompt to stop the interaction. This allows for immediate testing and exploration of the RAG system's performance on your own data.

In [ ]:
# Start an infinite loop to allow for continuous interaction with the RAG system.
while True:                                                                      # ask anything, 'quit' to stop
    # Prompt the user to enter a question and remove leading/trailing whitespace.
    q = input("Ask your documents: ").strip()

    # Check if the user input is a command to quit (e.g., 'quit', 'exit', 'q', or empty).
    # If so, break out of the loop to end the interactive session.
    if q.lower() in ("quit", "exit", "q", ""): break

    # Call the `answer` function with the user's question and print the generated response.
    # A newline character `\n` is added for better readability between answers.
    print(answer(q), "\n")

If a question you *know* is answered comes back as "I don't have that information", the best similarity was below the threshold: your wording differs from the document's. Rephrase, or lower `threshold=0.25`. If the answer is wrong, print `search(kb, q)` — nine times out of ten the retrieval, not the model, is the problem.

## Exercises · Recap

These exercises are designed to deepen your understanding of RAG components and encourage further experimentation. They explore different aspects of chunking, retrieval, and LLM interaction, allowing you to test assumptions and optimize the system for various scenarios.

1.  **Overlap**: write `chunk_overlap(text, size=300, overlap=60)` (each chunk repeats the last 80 characters of the previous one), add it to `STRATEGIES`, re-run the table. Does it rescue the answers `tiny` lost?
2.  **k**: re-run `evaluate(kb, k=1)` and `k=6`. When does more context help, when is it noise?
3.  **Follow-ups**: ask *"And in the Netherlands?"* after the pension question — it fails. Fix it with one `chat()` call that rewrites the follow-up into a standalone question first.
4.  **Keyword + vector**: ChromaDB also has `where_document={"contained_by": "pension"}`. Combine it with the vector search for questions that need an exact word (a product name, an error code).

Here's a summary table comparing the components used in this lab with potential production-grade upgrades:

| Component | In this lab | Production upgrade |
|---|---|---|
| Loader + metadata | `fetch()` from the handbook API | connectors (SharePoint, Confluence, tickets), access control in metadata |
| Cleaner | `clean()` — 3 regexes | format-aware parsers (PDF layout, tables) |
| Chunker | `chunk_fixed` / `chunk_sections` + **measured** | parent–child chunks, overlap, per-document-type rules |
| Embeddings + vector DB | `all-MiniLM-L6-v2` + ChromaDB | bigger embedder, hybrid keyword + vector search, reranker |
| Retriever | `search()` with `where` filters | filters from the user's identity and question (country, date, permissions) |
| Generator + gate | `answer()` with a measured threshold | evaluation set: hit rate, faithfulness, answer quality |

*Handbook text © GitLab Inc., [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/).* **Next → Lab 2:** the *agent* decides which tool to call.